In [1]:
%%bash
set -uo pipefail
W=/kaggle/working
T=/tmp/repro
MAIN=$T/glassbox-mech
PY=$T/py/bin/python
REST='"numpy==1.26.4" "scipy==1.17.1" "einops==0.8.2" "transformer-lens==2.17.0" "transformers==4.57.6" "pyarrow==23.0.0" "datasets==4.5.0" "pandas==3.0.0" pytest'
rm -rf $T $W/reports && mkdir -p $T $W/reports
git clone -q https://github.com/designer-coderajay/glassbox-mech.git $MAIN
git -C $MAIN log -1 --format='%H %s' | tee $W/reports/main.commit.txt
pip install -q uv
uv venv $T/py --python 3.11 -q
uv pip install -q --python $PY --index-url https://download.pytorch.org/whl/cpu "torch==2.10.0"
eval uv pip install -q --python $PY $REST
uv pip install -q --python $PY -e $MAIN --no-deps
uv pip freeze --python $PY > $W/reports/env_freeze.txt
lscpu | head -20 | tee $W/reports/lscpu.txt
df -h /tmp $W | tee $W/reports/disk_start.txt
CPU="$(lscpu | grep 'Model name' | sed 's/Model name:[[:space:]]*//') / Linux x86-64 / Kaggle CPU"
prep () {
  git -C $MAIN worktree add -q $T/wt/$1 $2
  git -C $T/wt/$1 log -1 --format='%H %s' > $W/reports/$1.commit.txt
  ln -s $T/py $T/wt/$1/.venv-repro
  uv pip install -q --python $PY -e $T/wt/$1 --no-deps
}
cmp () {
  p=$1; shift
  uv pip install -q --python $PY -e $MAIN --no-deps
  mkdir -p $W/reports/$p.fresh
  for f in "$@"; do cp "$f" $W/reports/$p.fresh/$(echo "$f" | sed "s|$T/wt/$p/||; s|/|__|g") 2>/dev/null; done
  (cd $MAIN && $PY -m glassbox.v7.reproduce compare evidence/$p "$@" --reproducer "Ajay Pravin Mahale (author)" --environment "$CPU" --out $W/reports/$p.json) 2>&1 | tee $W/reports/$p.compare.txt
  git -C $MAIN worktree remove --force $T/wt/$p
  df -h /tmp $W | tail -2
}

prep v7-exp4-interventions ad75d43
(cd $T/wt/v7-exp4-interventions && .venv-repro/bin/python experiments/v7/intervention/run.py --out repro/exp4) > $W/reports/v7-exp4-interventions.log 2>&1
cmp v7-exp4-interventions $T/wt/v7-exp4-interventions/repro/exp4/results.json

prep v7-exp3b-noise-baseline-merge 55e8bcf
(cd $T/wt/v7-exp3b-noise-baseline-merge && .venv-repro/bin/python experiments/v7/noise_baseline_3b/run.py --out repro/exp3b) > $W/reports/v7-exp3b-noise-baseline-merge.log 2>&1
cmp v7-exp3b-noise-baseline-merge $T/wt/v7-exp3b-noise-baseline-merge/repro/exp3b/results.json

for v in 0.14.17 0.14.19; do uv venv ~/.glassbox-venvs/li-$v --python 3.11 -q; uv pip install -q --python ~/.glassbox-venvs/li-$v/bin/python "llama-index-core==$v"; done
prep v7-exp2-llama-dedup 72cb710
(cd $T/wt/v7-exp2-llama-dedup && for v in 0.14.17 0.14.19; do for p in sync async; do for r in r1 r2; do ~/.glassbox-venvs/li-$v/bin/python experiments/v7/llama_dedup/subject.py --path $p --run $r --out repro/exp2; done; done; done && .venv-repro/bin/python experiments/v7/llama_dedup/evaluate.py --results repro/exp2) > $W/reports/v7-exp2-llama-dedup.log 2>&1
cmp v7-exp2-llama-dedup $T/wt/v7-exp2-llama-dedup/repro/exp2/results.json

uv venv ~/.glassbox-venvs/exp8-stack-b --python 3.11 -q
uv pip install -q --python ~/.glassbox-venvs/exp8-stack-b/bin/python "llama-index-core==0.14.19" "openinference-instrumentation-llama-index==4.6.0" "opentelemetry-sdk==1.45.1" "opentelemetry-exporter-otlp-proto-common==1.45.1" "opentelemetry-proto==1.45.1"
prep v7-exp8-cross-stack 226a4eb
(cd $T/wt/v7-exp8-cross-stack && ~/.glassbox-venvs/exp8-stack-b/bin/python experiments/v7/cross_stack/subject.py record --out repro/exp8 && ~/.glassbox-venvs/exp8-stack-b/bin/python experiments/v7/cross_stack/subject.py schema-check --out repro/exp8 && .venv-repro/bin/python experiments/v7/cross_stack/evaluate.py --results repro/exp8) > $W/reports/v7-exp8-cross-stack.log 2>&1
cmp v7-exp8-cross-stack $T/wt/v7-exp8-cross-stack/repro/exp8/results.json

prep v7-exp1-rag-fault aedfebf
(cd $T/wt/v7-exp1-rag-fault && .venv-repro/bin/python experiments/v7/rag_fault/run.py --out repro/exp1) > $W/reports/v7-exp1-rag-fault.log 2>&1
cmp v7-exp1-rag-fault $T/wt/v7-exp1-rag-fault/repro/exp1/results.json

df -h /tmp $W | tee $W/reports/disk_end.txt
cd $W && zip -qr repro_A.zip reports && ls -la reports && grep -h VERDICT reports/*.compare.txt

1bb823421f4ff790d5b53eb7333426718c96bebd prereg(v7): Exp 9 addendum 1, attempt 1 stopped by full disk; CPU torch, shared venv, /tmp worktrees for attempt 2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.9/17.9 MB 73.9 MB/s eta 0:00:00
Architecture:                            x86_64
CPU op-mode(s):                          32-bit, 64-bit
Address sizes:                           46 bits physical, 48 bits virtual
Byte Order:                              Little Endian
CPU(s):                                  4
On-line CPU(s) list:                     0-3
Vendor ID:                               GenuineIntel
Model name:                              Intel(R) Xeon(R) CPU @ 2.20GHz
CPU family:                              6
Model:                                   79
Thread(s) per core:                      2
Core(s) per socket:                      2
Socket(s):                               1
Stepping:                                0
BogoMIPS:                                4400.33
Flags:   

Using Python 3.11.17 environment at: /tmp/repro/py
